# ðŸ›’ Supermarket Sales Analysis

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

**Objective:** Analyze supermarket sales data to uncover insights about customer behavior, product performance, and revenue trends.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
%pip install --upgrade --force-reinstall matplotlib
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('Libraries loaded successfully âœ…')

ModuleNotFoundError: No module named 'matplotlib.backends.registry'

## 2. Load Dataset

We'll use the **Supermarket Sales** dataset. If you don't have the CSV, download it from [Kaggle](https://www.kaggle.com/datasets/aungpyaeap/supermarket-sales).

In [ ]:
# Load the dataset
df = pd.read_csv('supermarket_sales.csv')

# Preview the first 5 rows
df.head()

> **Note:** If you don't have the CSV, you can generate a synthetic dataset below:

In [ ]:
# --- OPTIONAL: Generate synthetic data if CSV is missing ---
np.random.seed(42)
n = 1000
cities = ['Yangon', 'Mandalay', 'Naypyitaw']
genders = ['Male', 'Female']
prod_lines = ['Health and beauty', 'Electronic accessories', 'Home and lifestyle',
              'Sports and travel', 'Food and beverages', 'Fashion accessories']
payments = ['Ewallet', 'Cash', 'Credit card']

df = pd.DataFrame({
    'Invoice ID': [f'{i:03d}-{np.random.randint(10,99)}-{np.random.randint(1000,9999)}' for i in range(n)],
    'Branch': np.random.choice(['A','B','C'], n),
    'City': np.random.choice(cities, n),
    'Customer type': np.random.choice(['Member','Normal'], n),
    'Gender': np.random.choice(genders, n),
    'Product line': np.random.choice(prod_lines, n),
    'Unit price': np.round(np.random.uniform(10, 100, n), 2),
    'Quantity': np.random.randint(1, 11, n),
    'Tax 5%': np.round(np.random.uniform(1, 50, n), 2),
    'Total': np.round(np.random.uniform(50, 1000, n), 2),
    'Date': pd.date_range('2019-01-01', periods=n, freq='H').astype(str),
    'Time': [f'{np.random.randint(10,21)}:{np.random.randint(0,60):02d}' for _ in range(n)],
    'Payment': np.random.choice(payments, n),
    'Rating': np.round(np.random.uniform(4, 10, n), 1)
})
df['gross income'] = np.round(df['Total'] * 0.05, 2)
df.to_csv('supermarket_sales.csv', index=False)
print(f'Synthetic dataset created with {df.shape[0]} rows and {df.shape[1]} columns.')

## 3. Data Understanding

In [ ]:
# Shape of the dataset
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

In [ ]:
# Column info
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Check for duplicates
print(f'Duplicate rows: {df.duplicated().sum()}')

## 4. Data Cleaning & Feature Engineering

In [ ]:
# Remove duplicates
df = df.drop_duplicates()

# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Create new features
df['Month'] = df['Date'].dt.month_name()
df['Day'] = df['Date'].dt.day_name()
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M', errors='coerce').dt.hour

df[['Date', 'Month', 'Day', 'Hour']].head()

## 5. Exploratory Data Analysis (EDA)

### 5.1 Revenue by City

In [ ]:
city_sales = df.groupby('City')['Total'].sum().sort_values(ascending=False)
print(city_sales)

city_sales.plot(kind='bar', color=['#4C72B0','#DD8452','#55A868'])
plt.title('Total Sales by City')
plt.ylabel('Total Sales')
plt.xticks(rotation=0)
plt.show()

### 5.2 Product Line Performance

In [ ]:
prod_sales = df.groupby('Product line')['Total'].sum().sort_values(ascending=False)
print(prod_sales)

sns.barplot(x=prod_sales.values, y=prod_sales.index, palette='viridis')
plt.title('Total Sales by Product Line')
plt.xlabel('Total Sales')
plt.show()

### 5.3 Customer Type & Gender Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['Customer type'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[0],
                                          colors=['#66b3ff','#ff9999'])
axes[0].set_title('Customer Type Distribution')
axes[0].set_ylabel('')

df['Gender'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[1],
                                  colors=['#99ff99','#ffcc99'])
axes[1].set_title('Gender Distribution')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

### 5.4 Payment Method Analysis

In [ ]:
payment = df.groupby('Payment')['Total'].sum().sort_values(ascending=False)
print(payment)

sns.barplot(x=payment.index, y=payment.values, palette='coolwarm')
plt.title('Sales by Payment Method')
plt.ylabel('Total Sales')
plt.show()

### 5.5 Monthly Sales Trend

In [ ]:
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
monthly = df.groupby('Month')['Total'].sum().reindex(
    [m for m in month_order if m in df['Month'].unique()])

monthly.plot(kind='line', marker='o', color='teal')
plt.title('Monthly Sales Trend')
plt.ylabel('Total Sales')
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

### 5.6 Sales by Day of Week

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily = df.groupby('Day')['Total'].sum().reindex(
    [d for d in day_order if d in df['Day'].unique()])

daily.plot(kind='bar', color='coral')
plt.title('Sales by Day of Week')
plt.ylabel('Total Sales')
plt.xticks(rotation=45)
plt.show()

### 5.7 Rating Distribution

In [ ]:
sns.histplot(df['Rating'], bins=20, kde=True, color='purple')
plt.title('Customer Rating Distribution')
plt.xlabel('Rating')
plt.show()

print(f'Average Rating: {df["Rating"].mean():.2f}')

### 5.8 Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

### 5.9 Gross Income by Product Line

In [ ]:
income = df.groupby('Product line')['gross income'].sum().sort_values(ascending=False)
sns.barplot(x=income.values, y=income.index, palette='magma')
plt.title('Gross Income by Product Line')
plt.xlabel('Gross Income')
plt.show()

## 6. Key Insights & Summary

In [ ]:
print('=' * 50)
print('ðŸ“Š SUPERMARKET SALES â€” KEY INSIGHTS')
print('=' * 50)
print(f'ðŸ’° Total Revenue          : ${df["Total"].sum():,.2f}')
print(f'ðŸ“ˆ Total Gross Income     : ${df["gross income"].sum():,.2f}')
print(f'ðŸ§¾ Average Order Value    : ${df["Total"].mean():,.2f}')
print(f'ðŸ† Top City               : {city_sales.idxmax()} (${city_sales.max():,.2f})')
print(f'ðŸ›ï¸ Top Product Line       : {prod_sales.idxmax()} (${prod_sales.max():,.2f})')
print(f'ðŸ’³ Most Used Payment      : {payment.idxmax()}')
print(f'â­ Average Rating         : {df["Rating"].mean():.2f}')
print(f'ðŸ‘¥ Member Share           : {(df["Customer type"] == "Member").mean() * 100:.1f}%')
print('=' * 50)

## 7. Conclusion

### ðŸ” Summary of Findings

1. **City Performance** â€” One city consistently outperforms the others in total revenue.
2. **Product Lines** â€” Food & Beverages and Fashion Accessories typically lead sales.
3. **Customer Behavior** â€” Members and non-members both contribute significantly; there is room to boost membership.
4. **Payment Preferences** â€” E-wallet and credit card dominate, suggesting digital-first customers.
5. **Ratings** â€” Average rating around 7, indicating generally satisfied customers.

### ðŸ’¡ Business Recommendations

- ðŸ“Œ Launch **loyalty campaigns** to convert Normal customers into Members.
- ðŸ“Œ Increase stock for **top-performing product lines**.
- ðŸ“Œ Focus marketing on the **top-performing city** and improve underperformers.
- ðŸ“Œ Promote **e-wallet offers** to align with modern payment habits.

---

### ðŸš€ Next Steps

- Build a **sales forecasting model** (ARIMA / Prophet).
- Create an interactive **dashboard** with Plotly or Streamlit.
- Perform **customer segmentation** using K-Means clustering.

*End of Notebook* ðŸŽ‰